# Importing modules and settings

### Importing libraries

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns
import os

General settings of Scanpy

In [ ]:
sc.settings.verbosity = 3 
sc.logging.print_header()
sc.settings.set_figure_params(dpi=80, facecolor='white')

In [ ]:
umap_cmap = sns.blend_palette(['xkcd:light grey', 'xkcd:indigo'], as_cmap = True)

# Creating folders for output files (using os)

In [ ]:
# Output current working directory
os.getcwd()

In [ ]:
# Create the output folders
output_folder = 'outputs_250523'
marker_excel_folder = output_folder + '/marker_excel_folder'
marker_plot_folder = output_folder + '/marker_plot_folder'
count_excel_folder = output_folder + '/count_excel_folder'


In [ ]:
# Check if output folder exists
if not os.path.exists(output_folder):
    os.makedirs(output_folder)
    os.makedirs(marker_excel_folder)
    os.makedirs(marker_plot_folder)
    os.makedirs(count_excel_folder)
    
    print(f"Folder '{output_folder}' and internal folders created successfully")
else:
    print(f"Folder '{output_folder}' already exists")

# Declaring the input file

In [ ]:
# Read the h5ad file 
adata = sc.read_h5ad('./Smed_L78-L47_20250523_Results.h5ad')

In [ ]:
adata

In [ ]:
# Add the names of each Leiden resolution to a list
leiden_names = adata.obs.columns[adata.obs.columns.str.contains('leiden')].to_list()

In [ ]:
leiden_names

In [ ]:
# define the samp variable to select the sample type
samp = 'Sample'

# Writing excel files with wilcoxon and log reg markers for each resolution

In [ ]:
# Write excel files for each Leiden resolution with the wilcox method
for lei in leiden_names:
    markers_w = pd.DataFrame(adata.uns['rank_genes_groups_wilcox_'+ lei]['names']).head(50)
    markers_w_l = pd.DataFrame(adata.uns['rank_genes_groups_wilcox_'+ lei ]['pvals_adj']).head(50)
    with pd.ExcelWriter(marker_excel_folder + '/' + lei + '_markers_wilcoxon.xlsx') as writer:
        for col in markers_w.columns:
            df = adata.raw.var.loc[markers_w[col][markers_w_l[col] < 0.05].to_list()][['gene_type','gene_ddv6', 'Preferred_name','Description.x']]
            df.to_excel(writer, sheet_name='Cluster '+ col)

In [ ]:
# Write excel files for each Leiden resolution with the logreg method
for lei in leiden_names:
    markers_l = pd.DataFrame(adata.uns['rank_genes_groups_logreg_'+ lei]['names']).head(50)
    with pd.ExcelWriter(marker_excel_folder + '/' + lei +'_markers_logreg.xlsx') as writer:
        for col in markers_l.columns:
            df = adata.raw.var.loc[markers_l[col].to_list()][['gene_type','gene_ddv6', 'Preferred_name','Description.x']]
            df.to_excel(writer, sheet_name='Cluster '+ col)

# Creating marker plots in pdf

In [ ]:
# Create a PDF for each cluster in each Leiden resolution 
def get_plots (clusteringlayer, cluster, li_markers):
    fig, axs = plt.subplots(3, 3, figsize = (15, 15))
    
    sc.pl.umap(adata, color= clusteringlayer, legend_loc = 'on data', groups = cluster, na_in_legend = False, size = 5, legend_fontsize = 7, title = clusteringlayer+' cluster '+cluster, show = False, ax = axs[0, 0])
    
    while len(li_markers) < 8:
        li_markers.append(None)

    gene01 = li_markers[0]
    gene02 = li_markers[1]
    gene10 = li_markers[2]
    gene11 = li_markers[3]
    gene12 = li_markers[4]
    gene20 = li_markers[5]
    gene21 = li_markers[6]
    gene22 = li_markers[7]

    #Row 0 first row
    sc.pl.umap(adata, color= gene01, title = gene01, color_map = umap_cmap, show = False, ax = axs[0, 1])
    sc.pl.umap(adata, color= gene02, title = gene02, color_map = umap_cmap, show = False, ax = axs[0, 2])
    
    #Row 1 second row
    
    sc.pl.umap(adata, color= gene10, title = gene10, color_map = umap_cmap, show = False, ax = axs[1, 0])
    sc.pl.umap(adata, color= gene11, title = gene11, color_map = umap_cmap, show = False, ax = axs[1, 1])
    sc.pl.umap(adata, color= gene12, title = gene12, color_map = umap_cmap, show = False, ax = axs[1, 2])
    
    #Row 2 third row
    
    sc.pl.umap(adata, color= gene20, title = gene20, color_map = umap_cmap, show = False, ax = axs[2, 0])
    sc.pl.umap(adata, color= gene21, title = gene21, color_map = umap_cmap, show = False, ax = axs[2, 1])
    sc.pl.umap(adata, color= gene22, title = gene22, color_map = umap_cmap, show = False, ax = axs[2, 2])

    return fig
    plt.close(fig)

In [ ]:
for lei in leiden_names:
    for i in adata.obs[lei].cat.categories:
        li = []
        lfc_s = pd.Series(adata.uns['rank_genes_groups_wilcox_'+lei]['logfoldchanges'][i])
        pval_s = pd.Series(adata.uns['rank_genes_groups_wilcox_'+lei]['pvals'][i])
        he_m = max(list(set(pval_s[pval_s < 0.05].index.to_list()) & set(lfc_s[lfc_s > 0].index.to_list())))
        wl = pd.DataFrame(adata.uns['rank_genes_groups_wilcox_'+lei]['names']).head(he_m)[i]
        lr = pd.DataFrame(adata.uns['rank_genes_groups_logreg_'+lei]['names']).head(30)[i]
        li = wl[wl.isin(lr)].to_list()
        figure = get_plots(lei, i, li)
        figure.savefig(marker_plot_folder + '/' + lei +'_cluster_'+ i +'.pdf', format = 'pdf')
        figure.clf()
        plt.close(figure)

# Writing excel files with counts per cluster and sample for each resolution

In [ ]:
# Create output excel files with the cell counts for each cluster and sample combination, for each leiden clustering resolution
counts_di = {}
for lei in leiden_names:
    series = []
    total = adata.obs[lei].value_counts().rename(lei + '_' + 'total')
    series.append(total)
    for sample in adata.obs[samp].cat.categories:
        partial = adata.obs[adata.obs[samp] == sample][lei].value_counts().rename(lei + '_' + sample)
        series.append(partial)
    counts_di[lei] = pd.concat(series, axis = 1)
    

In [ ]:
for key in counts_di.keys():
    counts_di[key].to_excel(count_excel_folder+  '/' + key + 'counts.xlsx')